<a href="https://colab.research.google.com/github/Halidh-Ahamed/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## Ranked actions + reason codes

The model output is converted into a ranked content action queue instead of making automatic decisions.

Pages are ordered by their predicted refresh probability. Each recommendation includes a priority level and simple reason codes that explain why the page was ranked highly.

This queue is intended to support editorial decision-making rather than replace human judgment.

In [8]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{}'
)
""".format(HF_TOKEN))

rel = "hf://datasets/FlyRank/internship-warehouse"

feature_df = con.sql(f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4,
    gsc_data_available
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""").df()

feature_df["avg_position"] = (
    feature_df["gsc_sum_position"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

feature_df["ctr"] = (
    feature_df["gsc_clicks"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4,gsc_data_available,avg_position,ctr
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,False,True,3.350000,0.000
1,2026-03-01,content_05597932fe4da067,1,0,0,True,False,True,0.000000,0.000
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,False,True,4.928000,0.008
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,False,True,4.000000,0.000
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,False,True,2.272727,0.000


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# ----------------------------
# Create target (same as Week 6)
# ----------------------------

feature_df["target"] = (
    feature_df["gsc_clicks"] > 0
).astype(int)

# ----------------------------
# Build modelling dataframe
# ----------------------------

model_df = feature_df[
    [
        "content_hash_id",
        "gsc_impressions",
        "avg_position",
        "ctr",
        "client_has_gsc",
        "client_has_ga4",
        "gsc_data_available",
        "target",
    ]
].dropna()

# ----------------------------
# Features / Target
# ----------------------------

X = model_df.drop(
    columns=["content_hash_id", "target"]
)

y = model_df["target"]

# ----------------------------
# Train/Test Split
# ----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# ----------------------------
# Train Random Forest
# ----------------------------

rf = RandomForestClassifier(
    n_estimators=30,
    max_depth=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)

rf.fit(X_train, y_train)

print("✅ Model rebuilt successfully!")

✅ Model rebuilt successfully!


In [10]:
# Create ranked action queue

action_df = feature_df.copy()

# Predict probability of needing attention
action_df["refresh_probability"] = rf.predict_proba(
    action_df[
        [
            "gsc_impressions",
            "avg_position",
            "ctr",
            "client_has_gsc",
            "client_has_ga4",
            "gsc_data_available",
        ]
    ]
)[:, 1]

# Rank highest probability first
action_df = action_df.sort_values(
    "refresh_probability",
    ascending=False,
)

# Reason codes
action_df["reason_code"] = ""

action_df.loc[
    action_df["ctr"] < 0.02,
    "reason_code"
] += "Low CTR; "

action_df.loc[
    action_df["avg_position"] > 8,
    "reason_code"
] += "Poor Avg Position; "

action_df.loc[
    action_df["gsc_impressions"] < 50,
    "reason_code"
] += "Low Impressions; "

action_df.loc[
    action_df["reason_code"] == "",
    "reason_code"
] = "General Review"

# Recommended action
action_df["recommended_action"] = "Review Content"

action_df.loc[
    action_df["ctr"] < 0.02,
    "recommended_action"
] = "Improve Title / Meta Description"

action_df.loc[
    action_df["avg_position"] > 8,
    "recommended_action"
] = "SEO Optimisation"

action_df.loc[
    action_df["gsc_impressions"] < 50,
    "recommended_action"
] = "Expand / Refresh Content"

# Show top recommendations
action_df[
    [
        "content_hash_id",
        "refresh_probability",
        "reason_code",
        "recommended_action",
    ]
].head(10)

,content_hash_id,refresh_probability,reason_code,recommended_action
2,content_7a105f548d9c6916,1.0,Low CTR;,Improve Title / Meta Description
9841377,content_66097d8adf63762f,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001410,content_0f5e75b5c1f882fe,1.0,Low CTR;,Improve Title / Meta Description
1001414,content_64b1124e7bc50f9d,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001418,content_b9a4856ce451f80d,1.0,Low CTR;,Improve Title / Meta Description
1001426,content_0ab94e17d436179c,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001430,content_f8d39dda05f1a125,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001431,content_57107ae15b182cbc,1.0,Low CTR;,Improve Title / Meta Description
1001432,content_be0e0dd336e5bf61,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001433,content_6cb3d0c3f101f62f,1.0,Low CTR; Poor Avg Position;,SEO Optimisation


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use

This model is designed as a decision-support tool for identifying content pages that may benefit from review or refresh. It ranks pages based on their predicted probability of requiring attention, allowing content teams to prioritize their work more efficiently.

### Intended Users

- SEO Analysts
- Content Writers
- Marketing Teams

### Limitations

- The model should not automatically update or publish content.
- Predictions are based only on the available historical search performance features.
- Important business context, seasonal trends, and editorial priorities are not included.
- Human review is required before taking any action.

### Appropriate Use

The ranked output should be treated as a recommendation queue rather than a final decision. Human reviewers should validate every suggested action before implementation.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Rules

Before acting on any recommendation, a human reviewer should verify:

- The page is still relevant to the business.
- The recommendation aligns with current SEO and content goals.
- Seasonal or temporary traffic changes are considered.
- The page is not intentionally receiving low traffic.
- Any content changes follow editorial guidelines.

### No-Go List (Do NOT Automate)

The model should **not** automatically:

- Publish or modify website content.
- Delete existing pages.
- Change page titles or meta descriptions without review.
- Redirect URLs.
- Make decisions based only on the prediction score.
- Replace human editorial judgement.

The model is intended only as a decision-support system that helps prioritize pages for manual review.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring

The model should be monitored regularly to ensure its predictions remain useful.

Suggested monitoring includes:

- Average prediction probability over time.
- Distribution of recommended actions.
- Percentage of reviewed pages that actually required updates.
- Changes in CTR and impressions after content refresh.

### Retrain Triggers

The model should be retrained when:

- New search performance data becomes available.
- SEO trends change significantly.
- Model performance noticeably decreases.
- New features become available.
- The content strategy changes substantially.

Retraining helps keep the recommendations relevant as search behavior evolves.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
from pathlib import Path

# Create output folder
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export ranked action queue
export_columns = [
    "content_hash_id",
    "refresh_probability",
    "reason_code",
    "recommended_action",
]

output_path = output_dir / "ranked_action_queue.csv"

action_df[export_columns].to_csv(output_path, index=False)

print("✅ Export completed!")
print(f"File saved to: {output_path}")

# Preview
action_df[export_columns].head(10)

✅ Export completed!
File saved to: work/outputs/ranked_action_queue.csv


,content_hash_id,refresh_probability,reason_code,recommended_action
2,content_7a105f548d9c6916,1.0,Low CTR;,Improve Title / Meta Description
9841377,content_66097d8adf63762f,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001410,content_0f5e75b5c1f882fe,1.0,Low CTR;,Improve Title / Meta Description
1001414,content_64b1124e7bc50f9d,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001418,content_b9a4856ce451f80d,1.0,Low CTR;,Improve Title / Meta Description
1001426,content_0ab94e17d436179c,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001430,content_f8d39dda05f1a125,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001431,content_57107ae15b182cbc,1.0,Low CTR;,Improve Title / Meta Description
1001432,content_be0e0dd336e5bf61,1.0,Low CTR; Poor Avg Position;,SEO Optimisation
1001433,content_6cb3d0c3f101f62f,1.0,Low CTR; Poor Avg Position;,SEO Optimisation


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.